# Checking Singh and Mantri (2024) against the public datasets

The paper reports accuracies for six classifiers on four public datasets but never says how it tested them. This
notebook checks what each dataset's label actually is and what the models score under a test setup that's stated
and can be repeated.

The paper took three datasets from UCI and the autism one from a Kaggle copy, which holds the same 704 rows as the UCI
file (only the two missing ages are written differently). The paper reports 703 and never says why. All the files in
`data/` are from UCI. The liver, dermatology and autism files were downloaded on
20 October 2026, the hepatitis files on 23 October and `dermatology.names` on 25 October.

Every model is scored with stratified ten-fold cross-validation repeated five times (50 scores per model).
Imputation and scaling are inside the pipeline, so they're fitted on the training folds only.

In [3]:
import itertools
import warnings
from pathlib import Path

import pandas as pd
from sklearn.base import clone
from sklearn.compose import make_column_transformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (GridSearchCV, RepeatedStratifiedKFold, cross_val_predict, cross_val_score,
                                     cross_validate, train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import KBinsDiscretizer, MinMaxScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## Data

The liver and dermatology files have no header. Dermatology's class is column 34, and `?` marks a missing age.
The autism file is ARFF, so the column names come from the `@attribute` lines.

The paper says it removed duplicate rows. These files have some, and they're kept here so the numbers are for the
files as published. The liver section checks what removing them changes.

In [4]:
liver = (pd.read_csv("data/ilpd.csv", header=None,
                     names=["age", "gender", "TB", "DB", "ALP", "ALT", "AST", "TP", "ALB", "AG", "patient"])
         .assign(gender=lambda d: (d["gender"] == "Male").astype(int)))

dermatology = pd.read_csv("data/dermatology.data", header=None, na_values="?")

lines = Path("data/autism.arff").read_text().splitlines()
columns = [line.split()[1] for line in lines if line.startswith("@attribute")]
rows = [line.split(",") for line in lines[lines.index("@data") + 1:] if line]
autism = pd.DataFrame(rows, columns=columns).replace("?", pd.NA)

for name, data in [("liver", liver), ("dermatology", dermatology), ("autism", autism)]:
    print(f"{name}: {len(data)} rows, {data.isna().any(axis=1).sum()} with missing values, "
          f"{data.duplicated().sum()} identical to an earlier row")

liver: 583 rows, 4 with missing values, 13 identical to an earlier row
dermatology: 366 rows, 8 with missing values, 0 identical to an earlier row
autism: 704 rows, 95 with missing values, 5 identical to an earlier row


## Models and folds

The paper says it filled missing values "using the standard deviation", which isn't a way to estimate a value, so
here they get the training median instead. The SVM also scales to 0-1, as the paper does. `baseline` always
answers the commonest class in the training folds, and every result should be compared with it.

Results are printed as the mean of the 50 fold scores +/- their standard deviation. That's the spread between
folds, not a confidence interval.

In [5]:
folds = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=0)
forest = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=0))
linear_svm = make_pipeline(SimpleImputer(strategy="median"), MinMaxScaler(), SVC(kernel="linear"))
baseline = DummyClassifier(strategy="most_frequent")  # ignores X, always answers the training folds' commonest class


def cv_accuracy(model, X, y):
    scores = cross_val_score(model, X, y, cv=folds, n_jobs=-1) * 100
    return f"{scores.mean():.1f} +/- {scores.std():.1f}"

## Hepatitis

Paper: RF 88.66%. The first column is whether the patient died (1) or lived (2), so the task is predicting
survival, not spotting hepatitis. The only model here is the baseline, because the point is what the label means,
not the accuracy.

In [6]:
hepatitis = pd.read_csv("data/hepatitis.data", header=None, na_values="?")
outcome = hepatitis[0].map({1: "DIE", 2: "LIVE"})

print("hepatitis outcome counts:", outcome.value_counts().to_dict())
print("always guess LIVE:", cv_accuracy(baseline, hepatitis.drop(columns=0), outcome))

hepatitis outcome counts: {'LIVE': 123, 'DIE': 32}
always guess LIVE: 79.4 +/- 2.3


## Autism

Paper: DT and RF 100%, linear SVM 92.15%. `result` is the total of the ten screening answers, and the label is
YES exactly when it's 7 or more, so a model given the answers only has to find the cut-off.

In [7]:
answers = autism[[f"A{i}_Score" for i in range(1, 11)]].astype(int)
score = autism["result"].astype(int)
asd = autism["Class/ASD"]

print("result is the sum of the ten answers in every row:", (answers.sum(axis=1) == score).all())
print(pd.crosstab(score, asd))
print("decision tree on the score alone:", cv_accuracy(DecisionTreeClassifier(), score.to_frame(), asd))

result is the sum of the ten answers in every row: True
Class/ASD   NO  YES
result             
0           14    0
1           33    0
2           74    0
3          110    0
4          131    0
5           83    0
6           70    0
7            0   57
8            0   55
9            0   47
10           0   30
decision tree on the score alone: 100.0 +/- 0.0


Dropping `result` doesn't help. A linear model can add up the answers itself.

In [8]:
print("linear SVM on the ten answers alone:", cv_accuracy(SVC(kernel="linear"), answers, asd))

linear SVM on the ten answers alone: 100.0 +/- 0.0


Without the answers, what's left describes who used the app. Column names are spelled as in the file. One age
is 383, so ages of 120+ are set to missing. Balanced accuracy is the average of the YES and NO hit rates, so
always answering NO only gets 50.

In [9]:
age = pd.to_numeric(autism["age"])
background = autism[["age", "gender", "ethnicity", "jundice", "austim", "contry_of_res", "used_app_before",
                     "relation"]].assign(age=age.where(age < 120))

categorical = list(background.columns.drop("age"))
preprocess = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), MinMaxScaler()), ["age"]),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categorical))

print("always guess NO:", cv_accuracy(baseline, background, asd))
for name, model in [("logistic regression", LogisticRegression(max_iter=2000)),
                    ("random forest", RandomForestClassifier(random_state=0))]:
    scores = cross_validate(make_pipeline(preprocess, model), background, asd, cv=folds,
                            scoring=["accuracy", "balanced_accuracy"], n_jobs=-1)
    accuracy = scores["test_accuracy"] * 100
    print(f"{name} without the answers: {accuracy.mean():.1f} +/- {accuracy.std():.1f}, "
          f"balanced accuracy {scores['test_balanced_accuracy'].mean() * 100:.1f}")

always guess NO: 73.2 +/- 0.4
logistic regression without the answers: 72.0 +/- 4.2, balanced accuracy 55.7
random forest without the answers: 70.3 +/- 4.8, balanced accuracy 58.7


On plain accuracy neither model beats always answering NO, though the gaps are smaller than the spread between
folds. On balanced accuracy both do a little better than 50, so these fields carry a weak signal. Either way,
the 100% comes from the answers and the cut-off, not from anything else in the file.

## Liver

Paper: RF 91.58%, linear SVM 90.38%. In the `patient` column 1 is a liver patient and 2 isn't (UCI page). About
71% of the file are patients, so that's what always answering "patient" scores. Four rows are missing the A/G
ratio, although the paper's Table 1 says the file has no missing values.

In [10]:
X_liver, y_liver = liver.drop(columns="patient"), liver["patient"]

print("always guess 'patient':", cv_accuracy(baseline, X_liver, y_liver))
print("random forest, all 10 features:", cv_accuracy(forest, X_liver, y_liver))
auc = cross_val_score(forest, X_liver, y_liver, cv=folds, scoring="roc_auc", n_jobs=-1).mean()
print("random forest, ranking quality (AUC):", round(auc, 2))
print("linear SVM, all 10 features:", cv_accuracy(linear_svm, X_liver, y_liver))

# cross_val_predict needs each row tested once, so plain 10-fold here
predictions = cross_val_predict(linear_svm, X_liver, y_liver, cv=10)
print("linear SVM predictions, out of fold:", pd.Series(predictions).value_counts().to_dict())

always guess 'patient': 71.4 +/- 0.7
random forest, all 10 features: 70.1 +/- 5.1
random forest, ranking quality (AUC): 0.74
linear SVM, all 10 features: 71.4 +/- 0.7
linear SVM predictions, out of fold: {1: 583}


Logistic regression is the model I'd use here, because it's easy to explain and gives a probability rather than a
yes/no. AUC is how well a model ranks patients above non-patients: 0.5 is a coin toss and 1 is perfect.

In [11]:
logistic = make_pipeline(SimpleImputer(strategy="median"), MinMaxScaler(), LogisticRegression(max_iter=2000))
scores = cross_validate(logistic, X_liver, y_liver, cv=folds, scoring=["accuracy", "roc_auc"], n_jobs=-1)
accuracy = scores["test_accuracy"] * 100
print(f"logistic regression, all 10 features: {accuracy.mean():.1f} +/- {accuracy.std():.1f}, "
      f"AUC {scores['test_roc_auc'].mean():.2f}")

logistic regression, all 10 features: 71.5 +/- 2.1, AUC 0.72


Identical rows can end up in both the training and test folds. Dropping them barely moves anything.

In [12]:
unique_liver = liver.drop_duplicates()
X_unique, y_unique = unique_liver.drop(columns="patient"), unique_liver["patient"]
print(f"without the {len(liver) - len(unique_liver)} duplicates, always guess 'patient':",
      cv_accuracy(baseline, X_unique, y_unique))
print("without duplicates, random forest:", cv_accuracy(forest, X_unique, y_unique))

without the 13 duplicates, always guess 'patient': 71.2 +/- 0.9
without duplicates, random forest: 68.9 +/- 4.1


The rough-set step kept 7 of the 10 features without saying which, so try all 120 subsets. Whichever 7 the paper
kept, it's one of these. This is for a default random forest on the raw values, and picking the best score after
seeing them all flatters it a little. Takes about ten minutes.

In [13]:
subset_scores = {
    subset: cross_val_score(forest, X_liver[list(subset)], y_liver, cv=folds, n_jobs=-1).mean() * 100
    for subset in itertools.combinations(X_liver.columns, 7)}
best = max(subset_scores, key=subset_scores.get)
print(f"best 7-feature subset of 120: {subset_scores[best]:.1f} {best}")

best 7-feature subset of 120: 72.8 ('gender', 'TB', 'DB', 'ALP', 'ALT', 'AST', 'ALB')


Two checks that the default settings aren't what's holding random forest back. Rough sets need values put into
bands, so one model bands each feature into five groups of equal size first (gender is already 0/1, so it's left alone). The other picks its own settings
with a small grid search inside each training fold.

In [14]:
# Many patients share the same low DB value, so some band edges coincide and sklearn merges them (and warns)
warnings.filterwarnings("ignore", message="Bins whose width are too small")
continuous = list(X_liver.columns.drop("gender"))
banding = make_column_transformer((KBinsDiscretizer(n_bins=5, encode="ordinal"), continuous), remainder="passthrough")
banded_forest = make_pipeline(SimpleImputer(strategy="median").set_output(transform="pandas"), banding,
                              RandomForestClassifier(random_state=0))
tuned_forest = GridSearchCV(forest, {"randomforestclassifier__min_samples_leaf": [1, 5, 10],
                                     "randomforestclassifier__max_features": ["sqrt", None]}, cv=5)
print("random forest on banded values:", cv_accuracy(banded_forest, X_liver, y_liver))
print("random forest, tuned inside each fold:", cv_accuracy(tuned_forest, X_liver, y_liver))

random forest on banded values: 70.0 +/- 5.4
random forest, tuned inside each fold: 69.2 +/- 4.7


One way to get a figure like the paper's: train on 80% and score on all the rows, so most of the test set was in
training. The paper doesn't say it did this. It only shows that a mistake like this can produce a number that
size. RF gains because it gets almost all of its training rows right. The SVM answers "patient" for everyone
either way.

In [15]:
X_train, _, y_train, _ = train_test_split(X_liver, y_liver, test_size=0.2, stratify=y_liver, random_state=0)
for name, model in [("random forest", forest), ("linear SVM", linear_svm)]:
    leaky = clone(model).fit(X_train, y_train).score(X_liver, y_liver) * 100
    print(f"{name} trained on 80%, scored on 100%:", round(leaky, 1))

random forest trained on 80%, scored on 100%: 94.9
linear SVM trained on 80%, scored on 100%: 71.4


## Dermatology

Paper: RF 97.29%, linear SVM 72.35%. The paper says there are 35 patients, but the file has 366.

Every patient has one of six skin diseases, and the model has to say which. There are 34 features: 12 from
examining the patient and 22 from a biopsy (see `data/dermatology.names`).

Psoriasis is the most common (112 of 366 patients). Always guessing psoriasis is right about 31% of the time, so
that's the baseline.

In [16]:
X_skin, y_skin = dermatology.drop(columns=34), dermatology[34]

print("always guess the commonest class:", cv_accuracy(baseline, X_skin, y_skin))
print("random forest, all 34 features:", cv_accuracy(forest, X_skin, y_skin))
print("linear SVM, all 34 features (paper: 72.35):", cv_accuracy(linear_svm, X_skin, y_skin))

always guess the commonest class: 30.6 +/- 1.0
random forest, all 34 features: 97.5 +/- 2.2
linear SVM, all 34 features (paper: 72.35): 97.5 +/- 2.2


Random forest scores 97.5%, which matches the paper. The linear SVM also scores 97.5%, not the 72.35% the paper
reports. So the two models are equally good here. The paper doesn't say how it set up or tested its models, so
there's no way to tell where its 25-point gap comes from. Either way, on this data random forest isn't the clear
winner the paper makes it.